# Electronics Retailer — EDA & Hypothesis Testing

This notebook pulls data straight from the MySQL star schema (via the `vw_sales_full` view
built in `Analysis code.sql`), explores it, and runs formal hypothesis tests —
the piece that differentiates this project from a plain dashboard.

**Note on this dataset:** it's synthetically generated, so don't be surprised if several
tests below come back *not* statistically significant (p > 0.05). That's a legitimate
finding to report — it tells you the data was generated without artificial bias baked in.
The point of this notebook is to show you can correctly set up, run, and interpret these
tests — not to manufacture dramatic findings.

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from sqlalchemy import create_engine
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)

## 1. Connect to MySQL and load the data

Pulling from `vw_sales_full` (created in `Analysis code.sql`) rather than
raw tables — the view already has the star schema joined and flattened.

In [ ]:
USER = "root"
PASSWORD = "ASD"
HOST = "localhost"
PORT = 3306
DB = "retail_star_schema"

engine = create_engine(f"mysql+pymysql://{USER}:{PASSWORD}@{HOST}:{PORT}/{DB}")

df = pd.read_sql("SELECT * FROM vw_sales_full", engine)
print(df.shape)
df.head()

## 2. Sanity checks

Before analyzing anything, confirm there's nothing broken in the pull — nulls where you
don't expect them, unexpected category values, etc.

In [ ]:
print(df.isnull().sum()[df.isnull().sum() > 0])
print()
print("Channels:", df.Channel.unique())
print("Loyalty tiers:", df.LoyaltyTier.unique())
print("Categories:", df.Category.unique())
print("Regions:", df.Region.unique())

*Expect `ShipDelayDays` and `ShipDate` to show a few nulls if any orders' ship dates fall outside the date dimension range — the extended `DimDate.csv` should have already covered this, but worth confirming here.*

## 3. Exploratory Data Analysis

### 3a. Revenue trend by year — is the business growing?

In [ ]:
yearly = df.groupby(df.OrderDate.astype('datetime64[ns]').dt.year)['SalesAmount'].sum()

yearly.plot(kind='bar', color='steelblue')
plt.title("Total Revenue by Year")
plt.ylabel("Revenue")
plt.xlabel("Year")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

yearly

### 3b. Revenue by category and region

Where is the money actually coming from?

In [ ]:
cat_region = df.pivot_table(index='Category', columns='Region', values='SalesAmount', aggfunc='sum')

plt.figure(figsize=(10, 6))
sns.heatmap(cat_region, annot=True, fmt='.0f', cmap='Blues')
plt.title("Revenue: Category x Region")
plt.tight_layout()
plt.show()

### 3c. Channel and loyalty tier distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

df.Channel.value_counts().plot(kind='bar', ax=axes[0], color='coral')
axes[0].set_title("Order Volume by Channel")
axes[0].tick_params(axis='x', rotation=0)

df.LoyaltyTier.value_counts().plot(kind='bar', ax=axes[1], color='seagreen')
axes[1].set_title("Order Volume by Loyalty Tier")
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

### 3d. Order value distribution by channel

A quick visual before we test it formally below.

In [ ]:
plt.figure(figsize=(9, 5))
sns.boxplot(data=df, x='Channel', y='SalesAmount')
plt.title("Order Value Distribution by Channel")
plt.tight_layout()
plt.show()

## 4. Hypothesis Test 1 — Does channel affect order value?

**H0:** Mean order value (`SalesAmount`) is the same for Online vs. Retail channels.
**H1:** Mean order value differs between Online vs. Retail channels.

Using **Welch's t-test** (`equal_var=False`) rather than a standard t-test, since we
shouldn't assume the two channels have equal variance without checking — Welch's is the
safer default.

In [ ]:
online = df.loc[df.Channel == 'Online', 'SalesAmount']
retail = df.loc[df.Channel == 'Retail', 'SalesAmount']

print(f"Online:  n={len(online)}, mean={online.mean():.2f}, std={online.std():.2f}")
print(f"Retail:  n={len(retail)}, mean={retail.mean():.2f}, std={retail.std():.2f}")

t_stat, p_value = stats.ttest_ind(online, retail, equal_var=False)
print(f"\nWelch's t-test: t = {t_stat:.3f}, p = {p_value:.4f}")

alpha = 0.05
if p_value < alpha:
    print(f"p < {alpha} → reject H0: order value significantly differs by channel.")
else:
    print(f"p >= {alpha} → fail to reject H0: no significant difference in order value between channels.")

## 5. Hypothesis Test 2 — Is product category independent of region?

**H0:** Category purchased and customer region are independent (no association).
**H1:** Category purchased and region are associated.

Using a **Chi-square test of independence** since both variables are categorical.

In [ ]:
contingency = pd.crosstab(df['Category'], df['Region'])
print(contingency)

chi2, p_value, dof, expected = stats.chi2_contingency(contingency)
print(f"\nChi2 = {chi2:.3f}, p = {p_value:.4f}, dof = {dof}")

if p_value < alpha:
    print(f"p < {alpha} → reject H0: category preference is associated with region.")
else:
    print(f"p >= {alpha} → fail to reject H0: category preference appears independent of region.")

## 6. Bonus — Does discount level vary by channel?

**H0:** Mean discount is the same across all four channels.
**H1:** At least one channel's mean discount differs.

Using **one-way ANOVA** since we're comparing more than two group means at once.

In [ ]:
groups = [df.loc[df.Channel == ch, 'Discount'] for ch in df.Channel.unique()]
f_stat, p_value = stats.f_oneway(*groups)

print(f"F = {f_stat:.3f}, p = {p_value:.4f}")
if p_value < alpha:
    print(f"p < {alpha} → reject H0: at least one channel's average discount differs significantly.")
else:
    print(f"p >= {alpha} → fail to reject H0: no significant discount difference across channels.")

## 7. Summary of findings

**Revenue trend:** Total revenue is essentially flat across all 7 years (~$10.0–10.2M annually,
2018–2024), with no meaningful upward or downward trend. Expected for a synthetically generated
dataset rather than one modeling organic growth or decline.

**Test 1 — Does sales channel affect order value?** (Welch's t-test)
- Online: n = 24,930, mean = 706.50
- Retail: n = 25,140, mean = 703.41
- t = 0.564, p = 0.57 → **fail to reject H0**
- *Takeaway:* Channel has no statistically significant effect on order value. Any decision about
  which channel to prioritize should rest on other factors (cost to serve, acquisition cost,
  margin) rather than order value.

**Test 2 — Is product category independent of region?** (Chi-square test of independence)
- χ² = 18.73, dof = 20, p = 0.54 → **fail to reject H0**
- *Takeaway:* Category mix doesn't meaningfully vary by region. No statistical case here for
  region-specific merchandising — a single national category strategy is sufficient.

**Test 3 — Does discount level vary by channel?** (One-way ANOVA)
- F = 1.76, p = 0.15 → **fail to reject H0**
- *Takeaway:* Discounting is applied consistently across channels rather than concentrated in one.

**Overall:** All three tests came back non-significant. For a synthetic dataset with no engineered
bias, that's the *correct* and expected result — not a failed analysis. The portfolio value here
is demonstrating rigorous method: the right test chosen for each variable type, clearly stated
H0/H1, and honest interpretation of p-values rather than manufacturing significance that isn't
in the data.